# Day 1: Hinglish ASR baselines (GPU T4)

**Settings → Accelerator: GPU T4 ×1, Internet: On.** If the data-prep notebook has finished, add its output as input and use the first cell's copy option. Otherwise the second cell downloads the test set (443 MB, a few minutes).

In [ ]:
!nvidia-smi -L
!pip install -q jiwer evaluate indic-transliteration

In [ ]:
%%writefile prepare_test.py
import json, os, random
import soundfile as sf

BASE = "data/test"
TR = f"{BASE}/transcripts"
OUT = "data/test_segments"
os.makedirs(OUT, exist_ok=True)

def read_map(path):
    d = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(" ", 1)
            if len(parts) == 2:
                d[parts[0]] = parts[1]
    return d

text = read_map(f"{TR}/text")
wavscp = read_map(f"{TR}/wav.scp")
utt2spk = read_map(f"{TR}/utt2spk")

segs = []
with open(f"{TR}/segments", encoding="utf-8") as f:
    for line in f:
        uid, rec, start, end = line.split()
        segs.append((uid, rec, float(start), float(end)))

cache, rows = {}, []
skipped = {"no_text": 0, "too_short": 0, "too_long": 0}
for uid, rec, start, end in segs:
    t = text.get(uid, "").strip()
    dur = end - start
    if not t:
        skipped["no_text"] += 1; continue
    if dur < 1.0:
        skipped["too_short"] += 1; continue
    if dur > 30.0:
        skipped["too_long"] += 1; continue
    if rec not in cache:
        cache[rec] = sf.read(os.path.join(BASE, wavscp[rec]))
    audio, sr = cache[rec]
    clip = audio[int(start * sr):int(end * sr)]
    path = f"{OUT}/{uid}.wav"
    sf.write(path, clip, sr)
    rows.append({"id": uid, "path": path, "text": t,
                 "speaker": utt2spk.get(uid, "?"), "duration": round(dur, 2)})

with open("data/test_all.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

random.seed(0)
subset = random.sample(rows, min(300, len(rows)))
with open("data/eval_300.jsonl", "w", encoding="utf-8") as f:
    for r in subset:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

hours = sum(r["duration"] for r in rows) / 3600
print(f"segments kept: {len(rows)} ({hours:.2f} h)  skipped: {skipped}")
print(f"speakers: {len(set(r['speaker'] for r in rows))}")
print(f"eval subset: {len(subset)} clips, "
      f"{len(set(r['speaker'] for r in subset))} speakers, "
      f"{sum(r['duration'] for r in subset)/60:.1f} min")


In [ ]:
%%writefile eval.py
"""Evaluate a Whisper checkpoint on a fixed jsonl split (MUCS 2021 Hinglish).

Corpus-level metrics:
  wer / cer                    Devanagari-safe normalization (NFC, lowercase, strip P*/S*)
  wer_romanized / cer_rom.     Devanagari words transliterated to ITRANS first
  pred_script_share            majority script of each prediction (devanagari/latin/arabic/other/empty)
  loop_share_pred / _ref       share of texts containing a repetition loop (heuristic; _ref = false-positive check)
  rtf                          wall time / audio time on --device, batched, after warm-up
"""
import argparse, json, os, re, time, unicodedata
from collections import Counter

import evaluate, jiwer, soundfile as sf, torch
from transformers import pipeline
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate


def norm(t):
    t = unicodedata.normalize("NFC", t).lower().replace("।", " ")
    # strip punctuation/symbols but KEEP Devanagari vowel signs (category M);
    # Whisper's BasicTextNormalizer deletes them
    t = "".join(" " if unicodedata.category(c)[0] in "PS" else c for c in t)
    return " ".join(t.split())


def is_deva(w):
    return any("\u0900" <= c <= "\u097f" for c in w)


# ITRANS has no mapping for candra vowels (ॉ ऑ ॅ ऍ), which Hindi uses mainly for
# English loanwords ("ऑफिस"); left alone they'd be stripped below ("phisa").
DEVA_FIX = str.maketrans({"ऑ": "ओ", "ॉ": "ो", "ऍ": "ए", "ॅ": "े"})


def romanize(t):
    words = []
    for w in norm(t).split():
        if is_deva(w):
            w = transliterate(w.translate(DEVA_FIX), sanscript.DEVANAGARI, sanscript.ITRANS)
            # word-final schwa deletion: ITRANS writes the inherent vowel as lowercase
            # "a" (long आ is "A"), so "करना"->"karanA" keeps it, "ऑफिस"->"ophisa"->"ophis"
            if len(w) > 2 and w[-1] == "a" and w[-2].lower() not in "aeiou":
                w = w[:-1]
            w = w.lower()
        words.append(re.sub(r"[^a-z0-9]", "", w))
    return " ".join(w for w in words if w)


ARABIC = [(0x0600, 0x06FF), (0x0750, 0x077F), (0xFB50, 0xFDFF), (0xFE70, 0xFEFF)]


def script_of(t):
    c = Counter()
    for ch in t:
        o = ord(ch)
        if 0x0900 <= o <= 0x097F:
            c["devanagari"] += 1
        elif any(a <= o <= b for a, b in ARABIC):
            c["arabic"] += 1
        elif ch.isascii() and ch.isalpha():
            c["latin"] += 1
        elif ch.isalpha():
            c["other"] += 1
    return c.most_common(1)[0][0] if c else "empty"


# a 2-30 char unit repeated 4+ times in a row ("लिविविविवि", "click click click click click")
LOOP = re.compile(r"(.{2,30}?)\1{3,}")


def has_loop(t):
    return bool(LOOP.search(t))


def share(labels):
    c = Counter(labels)
    return {k: round(v / len(labels), 4) for k, v in c.most_common()}


def utt_wer(r, p):
    return 1.0 if not p else jiwer.wer(r, p)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="openai/whisper-base")
    ap.add_argument("--data", default="data/eval_300.jsonl")
    ap.add_argument("--n", type=int, default=300)
    ap.add_argument("--device", default="cuda:0" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--threads", type=int, default=4, help="CPU only")
    ap.add_argument("--batch_size", type=int, default=8)
    ap.add_argument("--max_new_tokens", type=int, default=440,
                    help="decoder has 448 positions incl. 4 prompt tokens; Devanagari costs "
                         "several byte-BPE tokens per word, so a low cap truncates real output")
    ap.add_argument("--no_repeat_ngram", type=int, default=0, help="0 = off (plain greedy)")
    ap.add_argument("--tag", default=None, help="output prefix; defaults to the model name")
    args = ap.parse_args()

    tag = args.tag or args.model.strip("/").replace("/", "_")
    if args.device == "cpu":
        torch.set_num_threads(args.threads)
    dtype = torch.float16 if args.device.startswith("cuda") else torch.float32

    rows_all = [json.loads(l) for l in open(args.data, encoding="utf-8")][: args.n]
    # jiwer rejects empty references; drop them identically for every model
    rows = [r for r in rows_all if norm(r["text"]) and romanize(r["text"])]
    n_dropped = len(rows_all) - len(rows)

    asr = pipeline("automatic-speech-recognition", model=args.model,
                   device=args.device, torch_dtype=dtype)
    gen = {"language": "hindi", "task": "transcribe", "max_new_tokens": args.max_new_tokens}
    if args.no_repeat_ngram:
        gen["no_repeat_ngram_size"] = args.no_repeat_ngram

    def load(path):
        a, sr = sf.read(path, dtype="float32")
        assert sr == 16000 and a.ndim == 1, path
        return {"raw": a, "sampling_rate": sr}

    # warm-up on fresh dicts (the pipeline may pop keys from its inputs)
    asr([load(rows[0]["path"]), load(rows[1]["path"])], batch_size=2, generate_kwargs=gen)

    audio = [load(r["path"]) for r in rows]
    t0 = time.perf_counter()
    out = asr(audio, batch_size=args.batch_size, generate_kwargs=gen)
    elapsed = time.perf_counter() - t0

    preds = [o["text"] for o in out]
    refs = [r["text"] for r in rows]
    P, R = [norm(p) for p in preds], [norm(r) for r in refs]
    Pr, Rr = [romanize(p) for p in preds], [romanize(r) for r in refs]
    audio_sec = sum(r["duration"] for r in rows)

    wer, cer = evaluate.load("wer"), evaluate.load("cer")
    metrics = {
        "tag": tag, "model": args.model, "device": args.device, "dtype": str(dtype).split(".")[-1],
        "n": len(rows), "n_dropped_empty_ref": n_dropped,
        "audio_min": round(audio_sec / 60, 2),
        "decoding": {k: v for k, v in gen.items()} | {"batch_size": args.batch_size},
        "wer": round(wer.compute(predictions=P, references=R), 4),
        "cer": round(cer.compute(predictions=P, references=R), 4),
        "wer_romanized": round(wer.compute(predictions=Pr, references=Rr), 4),
        "cer_romanized": round(cer.compute(predictions=Pr, references=Rr), 4),
        "pred_script_share": share([script_of(p) for p in preds]),
        "ref_script_share": share([script_of(r) for r in refs]),
        "loop_share_pred": round(sum(map(has_loop, preds)) / len(preds), 4),
        "loop_share_ref": round(sum(map(has_loop, refs)) / len(refs), 4),
        "rtf": round(elapsed / audio_sec, 4),
    }

    os.makedirs("results", exist_ok=True)
    with open(f"results/{tag}_preds.jsonl", "w", encoding="utf-8") as f:
        for r, p, pn, rn, pr, rr in zip(rows, preds, P, R, Pr, Rr):
            f.write(json.dumps({
                "id": r["id"], "speaker": r["speaker"], "duration": r["duration"],
                "ref": r["text"], "pred": p,
                "wer": round(utt_wer(rn, pn), 4), "wer_romanized": round(utt_wer(rr, pr), 4),
                "pred_script": script_of(p), "loop": has_loop(p),
            }, ensure_ascii=False) + "\n")
    with open(f"results/{tag}_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(json.dumps(metrics, indent=2, ensure_ascii=False))
    for r, p in list(zip(refs, preds))[:5]:
        print("\nREF :", r, "\nPRED:", p)


if __name__ == "__main__":
    main()


In [ ]:
import glob, os, subprocess
src = glob.glob("/kaggle/input/**/data/eval_300.jsonl", recursive=True)
if src:   # prep notebook output attached: reuse its test segments
    d = os.path.dirname(src[0]); os.makedirs("data", exist_ok=True)
    subprocess.run(f"cp -r {d}/test_segments {d}/test_all.jsonl {d}/eval_300.jsonl data/", shell=True, check=True)
    print("copied from", d)
else:
    print("no prep input attached: run the next cell")

In [ ]:
%%bash
# only if the cell above said no prep input
[ -f data/eval_300.jsonl ] && exit 0
mkdir -p data_raw_test data
wget -q -O- https://www.openslr.org/resources/104/Hindi-English_test.tar.gz | tar xz -C data_raw_test
T=$(dirname $(dirname $(find data_raw_test -path '*transcripts/segments' | head -1)))
ln -sfn "$(realpath $T)" data/test && python prepare_test.py

In [ ]:
import json, hashlib
h = hashlib.md5("".join(json.loads(l)["id"] for l in open("data/eval_300.jsonl")).encode()).hexdigest()
LOCAL_MD5 = "2699562b2ab929891ecdd2f7cf0816e6"
print(h, "MATCH" if h == LOCAL_MD5 else "MISMATCH: fix before comparing any numbers")

## Baselines (fp16, greedy, batch 8)
The last run repeats whisper-base with `no_repeat_ngram_size=5`, to measure what the anti-loop decoding does.

In [ ]:
for m in ["tiny", "base", "small"]:
    !python eval.py --model openai/whisper-{m} --device cuda:0
!python eval.py --model openai/whisper-base --device cuda:0 --no_repeat_ngram 5 --tag whisper-base_nrng5

In [ ]:
import pandas as pd
rows = []
for f in sorted(glob.glob("results/*_metrics.json")):
    m = json.load(open(f))
    rows.append({k: m[k] for k in ["tag", "n", "wer", "cer", "wer_romanized", "cer_romanized",
                                   "loop_share_pred", "loop_share_ref", "rtf"]}
                | {"pred_" + k: v for k, v in m["pred_script_share"].items()})
pd.DataFrame(rows).set_index("tag").fillna(0)

In [ ]:
!zip -qr results_day1.zip results eval.py && ls -lh results_day1.zip  # no audio inside